# 01 データをながめてみよう:がんの遺伝子データ

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cogproject/stats-ml-bio-study/blob/main/notebooks/01_data_overview.ipynb)

**今日のゴール(30分)**:本物の研究データを開いて、「どんなデータか」を自分の目で確かめる。

### そもそも、どんなデータ?
- 私たちの体は細胞でできていて、細胞の中には **遺伝子**(体の設計図の一つ一つ)が約2万個あります。
- 遺伝子は、いつも全部が働いているわけではありません。よく働く遺伝子もあれば、ほとんど休んでいる遺伝子もあります。
- 「その遺伝子がどれくらい働いているか」を数字にしたものを **発現量(はつげんりょう)** と呼びます。
- この数字を、細胞の中の物質を読み取る機械でまとめて測る方法が **RNA-seq(アールエヌエー・シーク)** です。

今日のデータは、がんの患者さん **801人** について、遺伝子 **20,531個** の発現量を測った表です。
がんの種類は5つです。名前は、米国のがん研究プロジェクト TCGA が付けた略称で、英語の正式名の頭の文字を取っています。

| 略称 | 英語の正式名 | 日本語 |
|---|---|---|
| BRCA | **BR**east invasive **CA**rcinoma | 浸潤性乳がん |
| KIRC | **KI**dney **R**enal **C**lear cell carcinoma | 腎明細胞がん |
| COAD | **CO**lon **AD**enocarcinoma | 結腸腺がん |
| LUAD | **LU**ng **AD**enocarcinoma | 肺腺がん |
| PRAD | **PR**ostate **AD**enocarcinoma | 前立腺がん(腺がん) |

- **Carcinoma(カルシノーマ)**:体の表面や、内側の壁をおおう細胞(上皮)からできるがん
- **Adenocarcinoma(腺がん)**:上皮のがんのうち、粘液などを出す「腺」の細胞からできるもの
- **Invasive(浸潤性)**:できた場所の外の、まわりの組織に広がっていること
- **Clear cell(明細胞)**:顕微鏡で見ると、細胞の中が透きとおって白っぽく見えるタイプ

⚠️ 「BRCA1」「BRCA2」という遺伝子の名前もありますが、ここでの BRCA はがんの種類の名前で、別物です。

略称の出典(英語、米国国立がん研究所 GDC):https://gdc.cancer.gov/resources-tcga-users/tcga-code-tables/tcga-study-abbreviations

> 例えるなら「801人分の健康診断の結果表。ただし項目は2万個ある」。昔は数個ずつしか測れなかった遺伝子の働きを、RNA-seq は全部いっぺんに測れるので、こんなに多くなります。それぞれの人がどのがんの種類かも分かっています。
>
> 遺伝子についての公式の説明(英語、米国 NIH):https://www.genome.gov/genetics-glossary/Gene

出典:UCI Machine Learning Repository「Gene expression cancer RNA-Seq」(元データは米国の TCGA プロジェクト)
https://archive.ics.uci.edu/dataset/401/gene+expression+cancer+rna+seq

### 使い方
上のセルから順に、左の ▶ ボタンを押して実行します。

## 0. 準備:グラフに日本語を出せるようにする

In [ ]:
!pip -q install japanize-matplotlib
import japanize_matplotlib

## 1. データをダウンロードする(1〜2分かかります)
インターネットからデータの入った圧縮ファイルを取ってきて、開きます。

In [ ]:
import urllib.request, zipfile, tarfile, glob, os
os.makedirs("data", exist_ok=True)
url = "https://archive.ics.uci.edu/static/public/401/gene+expression+cancer+rna+seq.zip"
zip_path = "data/rnaseq.zip"
if not os.path.exists(zip_path):          # まだ持っていなければダウンロード
    urllib.request.urlretrieve(url, zip_path)
with zipfile.ZipFile(zip_path) as z:      # zip を開く
    z.extractall("data")
for tgz in glob.glob("data/**/*.tar.gz", recursive=True):  # 中の tar.gz も開く
    with tarfile.open(tgz) as t:
        t.extractall("data")
data_csv = glob.glob("data/**/data.csv", recursive=True)[0]
labels_csv = glob.glob("data/**/labels.csv", recursive=True)[0]
print("見つけたファイル:", data_csv, labels_csv)

## 2. 表を開いて、大きさを見る
`pandas`(パンダス)は、Python で表を扱う道具です。Excel のようなものだと思ってください。

In [ ]:
import pandas as pd
X = pd.read_csv(data_csv, index_col=0)              # 発現量の表
y = pd.read_csv(labels_csv, index_col=0)["Class"]   # がんの種類
print("表の大きさ(行, 列):", X.shape)
X.iloc[:5, :5]   # 左上の 5行×5列 だけ見せる

**読み方**:1行が患者さん1人、1列が遺伝子1個です。`gene_0` などは遺伝子の番号です(本当の名前はこのデータには入っていません)。

🤔 **考えてみよう**:行(人数)より列(遺伝子の数)のほうがずっと多いですね。人数より調べる項目のほうが多いデータは、ふつうの成績表とはかなり違います。これが遺伝子データの特徴で、あとで学ぶ「主成分分析」が役に立つ理由です。

## 3. がんの種類ごとに、何人いる?

In [ ]:
print(y.value_counts())
y.value_counts().plot(kind="bar", title="がんの種類ごとの人数");

🤔 人数がいちばん多いのはどれ?人数に差があることは、あとで比べるときに気をつける点です。

## 4. 数字はどんなふうに散らばっている?
**ヒストグラム**は、「どのくらいの値が、何個あるか」を棒の高さで表すグラフです。

In [ ]:
import matplotlib.pyplot as plt
vals = X.values.ravel()   # 表の数字を1列に並べる
print("いちばん小さい値:", vals.min(), " いちばん大きい値:", round(vals.max(), 1))
print("0 の割合:", round((vals == 0).mean() * 100, 1), "%")
plt.hist(vals, bins=100)
plt.title("すべての発現量の散らばり"); plt.xlabel("発現量"); plt.ylabel("個数");

**読み方**:左はしの 0 のところに高い棒があります。これは「ほとんど働いていない遺伝子」がたくさんあるということです。

ℹ️ このデータは、配布元が見やすく計算し直した(正規化した)数字です。機械から出たままの数字は、ごく一部がとても大きいので、ふつうは「log(ログ)」という計算で大きい数を縮めてから見ます。これは後の回で扱います。

## 5. 1つの遺伝子を、がんの種類ごとに比べる
**箱ひげ図**は、グループごとの数字の散らばりを比べるグラフです。箱の中の線が真ん中の値(中央値)、箱が半分の人が入る範囲です。

In [ ]:
top_gene = X.var().idxmax()   # 人によって値がいちばん大きく違う遺伝子を選ぶ
df = pd.DataFrame({"発現量": X[top_gene], "がんの種類": y})
df.boxplot(column="発現量", by="がんの種類")
plt.title(f"{top_gene} の発現量"); plt.suptitle(""); plt.ylabel("発現量");

🤔 **考えてみよう**:この遺伝子は、どのがんでよく働いている?

その差が「たまたま」なのか「本当の差」なのかを、数字で確かめる方法を **検定** といいます(後の回で学びます)。

## 6. 次回の準備:大事そうな遺伝子にしぼる
2万個は多すぎるので、**人によって値がよく変わる遺伝子**(=がんの種類の違いが出やすい遺伝子)を上から500個えらびます。
「人によってどれくらい違うか」を表す数字を **分散** といいます。

In [ ]:
var = X.var().sort_values(ascending=False)   # 分散が大きい順に並べる
plt.plot(var.values[:2000])
plt.title("遺伝子ごとの分散(大きい順に2000個)"); plt.xlabel("順位"); plt.ylabel("分散");
X_top = X[var.index[:500]]
print("500個にしぼった表の大きさ:", X_top.shape)

**読み方**:分散が大きい遺伝子は一部だけで、多くの遺伝子は人によってあまり変わりません。

## 今日のまとめ
- 遺伝子データは「人数 < 調べる項目」という形の表だった
- 値の多くは 0(休んでいる遺伝子)だった
- がんの種類によって、働き方がちがう遺伝子がありそうだ

**次回(02)**:遺伝子どうしの「似た動き」(相関)を調べます。「似た動きをする」=「片方が原因」ではない、という大事な話もします。

### もっと知りたい人へ(公式の説明)
- pandas 入門:https://pandas.pydata.org/docs/user_guide/10min.html
- matplotlib 入門:https://matplotlib.org/stable/tutorials/pyplot.html
- TCGA について(英語):https://www.cancer.gov/ccg/research/genome-sequencing/tcga